# Beat Aligned Dataset (Refactored)

This notebook is a slim runner that calls functions from `src/preprocessing/beat_aligned_dataset.py` (no local pipeline function declarations).

In [ ]:
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from src.preprocessing.beat_aligned_dataset import setup_logging, run_pipeline


In [ ]:
# Adjust to your environment if needed
DATA_ROOT = Path(r"C:/Xuexi26Spring/taiko-diffusion/sample_data")
UNPACKED_ROOT = DATA_ROOT / "unpacked"
INDEX_DIR = DATA_ROOT / "chart_index"
DATASET_DIR = DATA_ROOT / "beat_aligned_dataset"

CHART_SUMMARY_CSV = INDEX_DIR / "chart_build_summary.csv"
SEQUENCE_METADATA_CSV = DATASET_DIR / "sequence_metadata.csv"
OUTPUT_SUMMARY_CSV = DATASET_DIR / "sanity_check_summary.csv"
OUTPUT_TOKEN_FREQ_CSV = DATASET_DIR / "sanity_check_token_frequency.csv"


In [ ]:
setup_logging()
run_pipeline(
    unpacked_root=UNPACKED_ROOT,
    index_dir=INDEX_DIR,
    dataset_dir=DATASET_DIR,
)


In [ ]:
# Sanity check (same purpose as the original notebook, without local function definitions)
chart_df = pd.read_csv(CHART_SUMMARY_CSV)
seq_df = pd.read_csv(SEQUENCE_METADATA_CSV)

print(f"Charts in chart_build_summary.csv: {len(chart_df)}")
print(f"Rows in sequence_metadata.csv: {len(seq_df)}")
if "status" in chart_df.columns:
    print("\nChart build status:")
    print(chart_df["status"].value_counts(dropna=False))

succeeded_charts = chart_df[chart_df["status"] == "ok"].copy() if "status" in chart_df.columns else chart_df
print(f"\nSucceeded charts: {len(succeeded_charts)}")
print("\nSequence metadata columns:")
print(list(seq_df.columns))

chart_id_cols = [c for c in ["folder_id", "chart_base", "chart_id"] if c in seq_df.columns]
if chart_id_cols:
    group_cols = chart_id_cols[:2] if len(chart_id_cols) >= 2 else chart_id_cols[:1]
    seq_count_per_chart = seq_df.groupby(group_cols).size().reset_index(name="n_sequences")
    print("\nSequence count per chart summary:")
    print(seq_count_per_chart["n_sequences"].describe())

json_col_candidates = ["token_json_path", "tokens_json_path", "token_path", "tokens_path", "json_path"]
audio_col_candidates = ["audio_npz_path", "audio_path", "npz_path"]

token_col = next((c for c in json_col_candidates if c in seq_df.columns), None)
audio_col = next((c for c in audio_col_candidates if c in seq_df.columns), None)
if token_col is None:
    raise ValueError(f"Could not find token JSON path column. Available columns: {list(seq_df.columns)}")
if audio_col is None:
    raise ValueError(f"Could not find audio NPZ path column. Available columns: {list(seq_df.columns)}")

print(f"\nUsing token JSON column: {token_col}")
print(f"Using audio NPZ column: {audio_col}")

token_counter = Counter()
token_lengths = []
empty_sequence_count = 0
total_sequences_from_json = 0
token_json_failures = []

unique_token_paths = seq_df[token_col].dropna().astype(str).unique().tolist()
for i, token_path_str in enumerate(unique_token_paths, start=1):
    token_path = Path(token_path_str)
    try:
        with open(token_path, "r", encoding="utf-8") as f:
            token_json_obj = json.load(f)

        # Normalize structures used in the original notebook
        if isinstance(token_json_obj, list):
            if len(token_json_obj) == 0:
                seq_tokens_list = []
            elif isinstance(token_json_obj[0], list):
                seq_tokens_list = token_json_obj
            elif isinstance(token_json_obj[0], dict) and "tokens" in token_json_obj[0]:
                seq_tokens_list = [x.get("tokens", []) for x in token_json_obj]
            else:
                raise ValueError("Unsupported list-based token JSON structure")
        elif isinstance(token_json_obj, dict) and "sequences" in token_json_obj:
            seqs = token_json_obj["sequences"]
            if len(seqs) == 0:
                seq_tokens_list = []
            elif isinstance(seqs[0], list):
                seq_tokens_list = seqs
            elif isinstance(seqs[0], dict) and "tokens" in seqs[0]:
                seq_tokens_list = [x.get("tokens", []) for x in seqs]
            else:
                raise ValueError("Unsupported dict['sequences'] token JSON structure")
        elif isinstance(token_json_obj, dict) and "sequence_token_data" in token_json_obj:
            seqs = token_json_obj["sequence_token_data"]
            if len(seqs) == 0:
                seq_tokens_list = []
            elif isinstance(seqs[0], dict) and "tokens" in seqs[0]:
                seq_tokens_list = [x.get("tokens", []) for x in seqs]
            else:
                raise ValueError("Unsupported dict['sequence_token_data'] structure")
        else:
            raise ValueError("Unsupported token JSON format")

        total_sequences_from_json += len(seq_tokens_list)
        for tokens in seq_tokens_list:
            token_lengths.append(len(tokens))
            if len(tokens) == 0:
                empty_sequence_count += 1
            token_counter.update(tokens)

    except Exception as e:
        token_json_failures.append({"token_json_path": str(token_path), "error": str(e)})

    if i % 50 == 0 or i == len(unique_token_paths):
        print(f"Processed token JSONs: {i} / {len(unique_token_paths)}")

vocab = sorted(token_counter.keys())
vocab_size = len(vocab)

print("\nToken sanity:")
print(f"Unique token JSON files: {len(unique_token_paths)}")
print(f"Total sequences from token JSONs: {total_sequences_from_json}")
print(f"Vocabulary size: {vocab_size}")
print(f"Empty sequences: {empty_sequence_count}")
print(f"Token JSON failures: {len(token_json_failures)}")
if token_lengths:
    token_len_series = pd.Series(token_lengths)
    print("\nToken length summary:")
    print(token_len_series.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

audio_shapes = []
audio_nan_files = 0
audio_inf_files = 0
audio_failures = []
total_sequences_from_audio = 0

unique_audio_paths = seq_df[audio_col].dropna().astype(str).unique().tolist()
for i, audio_path_str in enumerate(unique_audio_paths, start=1):
    audio_path = Path(audio_path_str)
    try:
        obj = np.load(audio_path, allow_pickle=False)
        if "audio_sequences" in obj.files:
            arr = obj["audio_sequences"]
        elif "arr_0" in obj.files:
            arr = obj["arr_0"]
        elif "X" in obj.files:
            arr = obj["X"]
        elif "data" in obj.files:
            arr = obj["data"]
        elif len(obj.files) == 1:
            arr = obj[obj.files[0]]
        else:
            raise ValueError(f"Could not identify audio array key in {audio_path}. Keys: {obj.files}")

        audio_shapes.append(tuple(arr.shape))
        total_sequences_from_audio += int(arr.shape[0])

        if np.isnan(arr).any():
            audio_nan_files += 1
        if np.isinf(arr).any():
            audio_inf_files += 1

    except Exception as e:
        audio_failures.append({"audio_npz_path": str(audio_path), "error": str(e)})

    if i % 50 == 0 or i == len(unique_audio_paths):
        print(f"Processed audio NPZs: {i} / {len(unique_audio_paths)}")

print("\nAudio sanity:")
print(f"Unique audio NPZ files: {len(unique_audio_paths)}")
print(f"Total sequences from audio arrays: {total_sequences_from_audio}")
print(f"Distinct audio shapes: {sorted(set(audio_shapes))[:10]}")
print(f"Audio files with NaN: {audio_nan_files}")
print(f"Audio files with Inf: {audio_inf_files}")
print(f"Audio NPZ failures: {len(audio_failures)}")

print("\nCross-check:")
print(f"sequence_metadata rows         = {len(seq_df)}")
print(f"total sequences from token JSON = {total_sequences_from_json}")
print(f"total sequences from audio NPZ  = {total_sequences_from_audio}")

summary_rows = [
    {"metric": "n_chart_rows", "value": len(chart_df)},
    {"metric": "n_succeeded_charts", "value": len(succeeded_charts)},
    {"metric": "n_sequence_metadata_rows", "value": len(seq_df)},
    {"metric": "n_unique_token_json_files", "value": len(unique_token_paths)},
    {"metric": "n_unique_audio_npz_files", "value": len(unique_audio_paths)},
    {"metric": "n_total_sequences_from_token_json", "value": total_sequences_from_json},
    {"metric": "n_total_sequences_from_audio_npz", "value": total_sequences_from_audio},
    {"metric": "vocab_size", "value": vocab_size},
    {"metric": "empty_sequence_count", "value": empty_sequence_count},
    {"metric": "token_json_failures", "value": len(token_json_failures)},
    {"metric": "audio_npz_failures", "value": len(audio_failures)},
    {"metric": "audio_nan_files", "value": audio_nan_files},
    {"metric": "audio_inf_files", "value": audio_inf_files},
]

if token_lengths:
    summary_rows.extend([
        {"metric": "token_len_min", "value": int(np.min(token_lengths))},
        {"metric": "token_len_mean", "value": float(np.mean(token_lengths))},
        {"metric": "token_len_median", "value": float(np.median(token_lengths))},
        {"metric": "token_len_p90", "value": float(np.percentile(token_lengths, 90))},
        {"metric": "token_len_p95", "value": float(np.percentile(token_lengths, 95))},
        {"metric": "token_len_p99", "value": float(np.percentile(token_lengths, 99))},
        {"metric": "token_len_max", "value": int(np.max(token_lengths))},
    ])

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")

token_freq_df = pd.DataFrame(
    [{"token": token, "count": count} for token, count in token_counter.most_common()]
)
token_freq_df.to_csv(OUTPUT_TOKEN_FREQ_CSV, index=False, encoding="utf-8-sig")

print("\nSaved:")
print(OUTPUT_SUMMARY_CSV)
print(OUTPUT_TOKEN_FREQ_CSV)

if token_lengths:
    suspicious_long_threshold = np.percentile(token_lengths, 99)
    print(f"\nSuggested suspicious token length threshold (p99): {suspicious_long_threshold:.2f}")
if token_json_failures:
    print("\nToken JSON failures (first 10):")
    print(pd.DataFrame(token_json_failures).head(10))
if audio_failures:
    print("\nAudio NPZ failures (first 10):")
    print(pd.DataFrame(audio_failures).head(10))
